In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint

import brokenaxes as bax
from matplotlib import pyplot as plt

from privtrajgen import config as conf
from privtrajgen.plot.figures import create_bar_plot, cm_to_inch, pt_to_inch
from privtrajgen.plot.tables import *
from privtrajgen.utils import logger, printc
from privtrajgen.utils import mean_conf_half

In [ ]:
# Logging
log = logger.configure_root_loger(logging.INFO)


def logc(header: str, *args: Any, column_width: int = 60) -> None:
    log.info(f"{header:<{column_width}}" + "".join([str(arg) for arg in args]))

In [ ]:
# Configuration
OUTPUT_DIR = os.path.join(conf.BASE_DIR, 'img/paper/')
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FORMAT = "pdf"

In [ ]:
# ACM
column_width_pt = 241.14749  # 241.14749pt
page_width_pt = 506.295  # 506.295pt

column_width = pt_to_inch(column_width_pt)
page_width = pt_to_inch(page_width_pt)
thesis_width = None

fig_width = column_width
fig_height = cm_to_inch(3.5)

FONT_SIZE = 6
plt.rcParams.update({
    'font.size': FONT_SIZE,  # Global font size
    'axes.titlesize': FONT_SIZE,  # Set title font size to same as axis labels
    'axes.labelsize': FONT_SIZE,  # Increase axis labels font by 2
    'xtick.labelsize': FONT_SIZE - 1,  # Tick-label font size
    'ytick.labelsize': FONT_SIZE - 1,  # Tick-label font size
    'legend.fontsize': FONT_SIZE,  # Make the legend/label fonts a little smaller
    'figure.subplot.wspace': 0.05,  # Reduce space between subfigures
    'figure.subplot.hspace': 0.05,  # Reduce vertical space to 0.1
    'figure.figsize': (column_width, fig_height),  # Default figure size
    'lines.markersize': 3,  # Adjust marker size
    'lines.linewidth': 1.5,  # Adjust line width
    # Legend
    'legend.framealpha': 1,  # No transparency
    'legend.edgecolor': 'black',  # Black border
    'legend.fancybox': False,  # No fancy box
    'legend.handletextpad': 0.2,
    'legend.columnspacing': 0.8,
})

print(f"Width:\t\t\t{fig_width:.2f} inch\t=\t{fig_width * 2.54:.2f} cm")
print(f"Height:\t\t\t{fig_height:.2f} inch\t=\t{fig_height * 2.54:.2f} cm")
print(f"Aspect ratio:\t{fig_width / fig_height:.2f}")

In [ ]:
ablation_2p_cases = ['25p', '2p', '26p', '27p']  # Ablation I Porto
ablation_2g_cases = ['25g', '2g', '26g', '27g']  # Ablation I GeoLIfe
ablation_3_cases = ['28p', '7p', '29p', '30p', '31p']  # Ablation II
nodp_cases = ['1p', '1g', '3p']  # No DP
ablation_cases = ablation_2p_cases + ablation_2g_cases + ablation_3_cases + nodp_cases

In [ ]:
# Load the results
results = {}
results_jsd = {}
line1 = {}  # Ablation II: line1[eps_s] = SWD
line1_jsd = {}  # Ablation II: line1[eps_s] = JSD
line2 = {}  # Ablation II: line2[eps_s] = SWD
line2_jsd = {}  # Ablation II: line2[eps_s] = JSD
line3 = {}  # Ablation III: line2[eps_c] = SWD
line3_jsd = {}  # Ablation III: line2[eps_c] = JSD
for case in ablation_cases:
    # Get the configuration
    config = final_evaluation[case]

    # Load results
    all_res = load_results(config)

    # Get the SWD
    swd = get_metric_values(all_res, keys.SWD_DISTRIBUTION)
    results[case] = swd

    # Get the JSD
    jsd = get_metric_values(all_res, keys.JSD_DISTRIBUTION)
    results_jsd[case] = jsd

    # Get the epsilon
    if case in ablation_2p_cases:
        assert config.dp.dp  # Assert that DP-SGD was used
        eps_s = config.dp.target_epsilon
        line1[eps_s] = swd
        line1_jsd[eps_s] = jsd

    if case in ablation_2g_cases:
        assert config.dp.dp  # Assert that DP-SGD was used
        eps_s = config.dp.target_epsilon
        line2[eps_s] = swd
        line2_jsd[eps_s] = jsd

    if case in ablation_3_cases:
        assert config.cond.dp  # Assert that DP cond. info. was used
        eps_c = config.cond.epsilon
        line3[eps_c] = swd
        line3_jsd[eps_c] = jsd
print("Ablation Ip:")
pprint(line1)
print("Ablation Ig:")
pprint(line2)
print("Ablation II:")
pprint(line3)

In [ ]:
# Append no-dp cases at x = 1000
line1[200] = results['1p']
line2[200] = results['1g']
line3[200] = results['3p']

In [ ]:
def get_stats(results: Dict[str, np.ndarray]) -> Tuple[List[float], List[float], List[float]]:
    """Return x, y, and yerr for the given results. yerr is the 95% confidence interval.

    params: results: Dict[str, np.ndarray]: Dictionary of results with d[x] = [y1, y2, ...]
    returns: Tuple[List[float], List[float], List[float]]: x, y, yerr
    """
    x = []
    y = []
    yerr = []
    for key, values in results.items():
        x.append(key)
        mean, half = mean_conf_half(values)
        y.append(mean)
        yerr.append(half)
    return x, y, yerr


# Create Figure
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
# Lines
x, y, yerr = get_stats(line1)
printc("Ablation I Porto", y)
ablation1_line = ax.errorbar(x, y, yerr=yerr, label=r"Abl. I Porto  ($\varepsilon_s$)", marker='o', linestyle='-',
                             capsize=3)
x, y, yerr = get_stats(line2)
printc("Ablation I GeoLife", y)
ablation2_line = ax.errorbar(x, y, yerr=yerr, label=r"Abl. I GeoLife ($\varepsilon_s$)", marker='s', linestyle='-',
                             capsize=3)
x, y, yerr = get_stats(line3)
printc("Ablation II", y)
ablation2_line = ax.errorbar(x, y, yerr=yerr, label=r"Abl. II Porto ($\varepsilon_c$)", marker='D', linestyle='-',
                             capsize=3)
# Labels
xlabel = r"$\varepsilon$"
ylabel = "SWD"
ax.set_xlabel(xlabel, labelpad=-5)
ax.set_ylabel(ylabel, labelpad=-1)
# ticks
xticks = [5, 10, 20, 50, 100, 200]
xticklabels = [str(x) for x in xticks]
xticklabels[-1] = r"$\infty$"
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)
yticks = [1e-3, 1e-2, 1e-1]
ax.set_yticks(yticks)
# Legend (Over the plot, 3 columns)
ax.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.29), columnspacing=0.5, facecolor='white', framealpha=1,
          markerscale=0.6, handlelength=1.0)
# ax.legend(loc='upper right')
# Crop
plt.subplots_adjust(left=0.115, right=0.99, top=0.82, bottom=0.14)
# Save
plt.savefig(os.path.join(OUTPUT_DIR, f"ablation_eps.pdf"), pad_inches=0.05)
plt.savefig(os.path.join(OUTPUT_DIR, f"ablation_eps.png"), pad_inches=0.05)

## Bar Plots

In [ ]:
plt.rcParams.update({
    'font.size': FONT_SIZE,  # Global font size
    'axes.labelsize': FONT_SIZE,  # Increase axis labels font by 2
    'axes.titlesize': FONT_SIZE,  # Set title font size to same as axis labels
    'xtick.labelsize': FONT_SIZE - 2,  # Tick-label font size
    'ytick.labelsize': FONT_SIZE - 2,  # Tick-label font size
    'legend.fontsize': FONT_SIZE - 1,  # Make the legend/label fonts a little smaller
    'figure.subplot.wspace': 0.05,  # Reduce space between subfigures
    'figure.subplot.hspace': 0.05,  # Reduce vertical space to 0.1
    'figure.figsize': (column_width, fig_height),  # Default figure size
    # 'lines.markersize': 3,                  # Adjust marker size
    'lines.linewidth': 1,  # Adjust line width
    # Legend
    'legend.framealpha': 1,  # No transparency
    'legend.edgecolor': 'black',  # Black border
    'legend.fancybox': False,  # No fancy box
    'legend.handletextpad': 0.2,
    'legend.columnspacing': 0.8,
})

IDs = [
    # Group 1: DiffTraj
    [1, 2, 3, 4, 7, 8],
    # Group 2: UnetVAE
    [9, 10, 11, 12, 15, 16],
    # Group 3: UnetGAN
    [17, 18, 19, 20, 23, 24],
    # Group 4: PrivTrace
    [None, 40, None, None, None, None],
]
porto_ids = [[f"{i}p" if i is not None else None for i in group] for group in IDs]
geolife_ids = [[f"{i}g" if i is not None else None for i in group] for group in IDs]

porto_group_names = ["DiffTraj", "UnetVAE", "UnetGAN", "PT", ]

ab3_labels = [
    "Uncond (C)",
    "Uncond + DP-SGD (D)",
    "Cond (A)",
    "Cond + DP-SGD (B)",
    "DP-Cond (C)",
    "DP-Cond + DP-SGD (D)",
]

adjust = {"left": 0.1, "right": 0.995, "top": 0.695, "bottom": 0.21}
create_bar_plot(
    porto_ids, keys.SWD_DISTRIBUTION, group_names=porto_group_names, labels=ab3_labels,
    y_label="SWD", title="SWD on Porto", group_offset=-0.005, adjust=adjust,
    yticks=[0, 0.01], output_dir=OUTPUT_DIR)

adjust = {"left": 0.09, "right": 0.995, "top": 0.69, "bottom": 0.2}
create_bar_plot(
    porto_ids, keys.JSD_DISTRIBUTION, "JSD on Porto",
    group_names=porto_group_names, labels=ab3_labels,
    y_label="JSD", group_offset=-0.15, adjust=adjust,
    output_dir=OUTPUT_DIR)

adjust = {"left": 0.1, "right": 0.995, "top": 0.69, "bottom": 0.21}
create_bar_plot(
    geolife_ids, keys.SWD_DISTRIBUTION, group_names=porto_group_names, labels=ab3_labels,
    y_label="SWD", title="SWD on GeoLife", group_offset=-0.023, adjust=adjust,
    output_dir=OUTPUT_DIR)

adjust = {"left": 0.09, "right": 0.995, "top": 0.69, "bottom": 0.2}
create_bar_plot(
    geolife_ids, keys.JSD_DISTRIBUTION, "JSD on GeoLife",
    group_names=porto_group_names, labels=ab3_labels,
    y_label="JSD", group_offset=-0.17, adjust=adjust,
    output_dir=OUTPUT_DIR)

## Research Question 2 Plots

In [ ]:
plt.rcParams.update({
    'font.size': FONT_SIZE,  # Global font size
    'axes.labelsize': FONT_SIZE,  # Increase axis labels font by 2
    'axes.titlesize': FONT_SIZE,  # Set title font size to same as axis labels
    'xtick.labelsize': FONT_SIZE - 2,  # Tick-label font size
    'ytick.labelsize': FONT_SIZE - 2,  # Tick-label font size
    'legend.fontsize': FONT_SIZE - 1,  # Make the legend/label fonts a little smaller
    'figure.subplot.wspace': 0.05,  # Reduce space between subfigures
    'figure.subplot.hspace': 0.05,  # Reduce vertical space to 0.1
    'figure.figsize': (column_width, fig_height),  # Default figure size
    # 'lines.markersize': 3,                  # Adjust marker size
    'lines.linewidth': 1,  # Adjust line width
    # Legend
    'legend.framealpha': 1,  # No transparency
    'legend.edgecolor': 'black',  # Black border
    'legend.fancybox': False,  # No fancy box
    'legend.handletextpad': 0.2,
    'legend.columnspacing': 0.8,
})

# Show the bars order by threat model for DiffTraj and UnetGAN
labels = [
    "Uncond (C)",
    "DP-8-Stat (C)",
    "DP-Sample (C)",

    "Uncond + DP-SGD (D)",
    "DP-8-Stat+DP-SGD (D)",
    "DP-Sample+DP-SGD (D)",
]

IDs = [
    # Group 1: TM C DiffTraj
    [1, 5, 7, None, None, None],
    # Group 2: TM C UnetGAN
    [17, 21, 23, None, None, None],
    # Group 3: TM D DiffTraj
    [None, None, None, 2, 6, 8],
    # Group 4: TM D UnetGAN
    [None, None, None, 18, 22, 24],
]
porto_ids = [[f"{i}p" if i is not None else None for i in group] for group in IDs]
geolife_ids = [[f"{i}g" if i is not None else None for i in group] for group in IDs]

group_names = [
    "DiffTraj (C)", "UnetGAN (C)",
    "DiffTraj (D)", "UnetGAN (D)",
]

adjust = {"left": 0.1, "right": 0.99, "top": 0.695, "bottom": 0.21}
create_bar_plot(porto_ids, keys.SWD_DISTRIBUTION, group_names=group_names, labels=labels, y_label="SWD",
                title=r"SWD on Porto by Threat Model ($\varepsilon_c = 10$)", group_offset=-0.0035, adjust=adjust,
                yticks=[0, 0.01], output_dir=OUTPUT_DIR, legend_rows_first=True, filename="porto_swd_by_tm")

adjust = {"left": 0.1, "right": 0.99, "top": 0.69, "bottom": 0.21}
create_bar_plot(geolife_ids, keys.SWD_DISTRIBUTION, group_names=group_names, labels=labels, y_label="SWD",
                title=r"SWD on GeoLife by Threat Model ($\varepsilon_c = 10$)", group_offset=-0.023, adjust=adjust,
                output_dir=OUTPUT_DIR, legend_rows_first=True, filename="geolife_swd_by_tm")

## Ablation Study III

In [ ]:
plt.rcParams.update({
    'font.size': FONT_SIZE,  # Global font size
    'axes.titlesize': FONT_SIZE,  # Title font size
    'axes.labelsize': FONT_SIZE,  # X and Y axis labels
    'xtick.labelsize': FONT_SIZE - 2,  # X-axis tick labels
    'ytick.labelsize': FONT_SIZE - 2,  # Y-axis tick labels
    'legend.fontsize': FONT_SIZE - 1,  # Legend font size
    'figure.titlesize': FONT_SIZE,  # Figure title font size
    'figure.labelsize': FONT_SIZE,  # Figure label font size
})


def create_bar_plot_ablation_III(cases: List[List[str]], metric: str, title: str,
                                 adjust=Dict[str, float],
                                 filename: Optional[str] = None,
                                 group_names: Optional[List[str]] = None, labels: List[str] = None,
                                 y_label: Optional[str] = None,
                                 group_offset: float = -0.004,
                                 brokenaxis: Optional[Tuple[Tuple[float, float], Tuple[float, float]]] = None,
                                 ) -> None:
    """
    Create a bar plot for the given cases and metric.

    :param cases: List[List[str]]: List of cases to plot. Each inner list is a group of bars.
    :param metric: str: Metric to plot.
    :param title: str: Title of the plot.
    :param filename: Optional[str]: Filename to save the plot. Defaults to title.
    :param group_names: Optional[List[str]]: Names of the groups.
    :param labels: List[str]: Labels for each case within the group.
    """

    # Assert input consistency
    assert len(cases) > 0, "No cases to plot"
    assert group_names is None or len(cases) == len(group_names), "Number of cases and group names do not match"
    # Get length of each group
    group_size = len(cases[0])
    for group in cases:
        assert len(group) == group_size, "Groups must have the same size"
    assert labels is None or len(labels) == group_size, "Number of labels must match group size"

    # Create figure
    fig_width = pt_to_inch(column_width_pt)
    fig_height = cm_to_inch(3.5)
    # Break y-axis
    if brokenaxis is not None:
        fig = plt.figure(figsize=(fig_width, fig_height))
        ax = bax.brokenaxes(ylims=brokenaxis)
    else:
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # Get colors for the bars
    colors = plt.colormaps.get_cmap('tab10')
    # Get different textures for the bars. Start with no texture for the first case.
    textures = [None] + ['////', '\\\\\\\\', '||||', '----', 'xxxx', '++++', 'oooo', '****', '....', '++++']

    # Get results
    results = {}
    for case in cases:
        for c in case:
            config = final_evaluation[c]
            all_res = load_results(config)
            metric_values = get_metric_values(all_res, metric)
            results[c] = metric_values

    # Use 95% confidence interval as error bars

    # Bar locations: Print each group without spacing, and one bar width spacing between groups
    bar_width = 1.2
    group_width = group_size * bar_width
    group_spacing = bar_width

    # Get x locations. Leave one group spacing on each side
    x = np.arange(len(cases)) * (group_width + group_spacing)
    # x += group_spacing / 2

    # Plot bars
    ids = []
    x_ticks = []
    bars = []
    for j, group in enumerate(cases):
        for i, case in enumerate(group):
            # Get the results
            metric_values = results[case]
            mean, half = mean_conf_half(metric_values, 0.95)
            # Get the color and texture
            color = colors(i)
            texture = textures[i]
            # x location for the bar
            x_loc = x[j] + i * bar_width
            # Plot the bar
            bar = ax.bar(x_loc, mean, yerr=half, capsize=3, width=bar_width, color=color, hatch=texture, align='edge')
            bars.append(bar)
            ids.append(case.upper())
            x_ticks.append(x_loc + bar_width / 2)

    # Use ids as xticks
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(ids)

    # The Group names should be underneath the xticks (IDs) and centered
    if group_names:
        for i, name in enumerate(group_names):
            x_loc = x[i] + group_width / 2
            ax.text(x_loc, group_offset, name, ha='center', va='center', fontsize=FONT_SIZE)

    # Legend
    legend_y = 1.31
    if labels:
        handles = [bars[i] for i in range(group_size)]
        ax.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, legend_y))

    # Y-Label = Metric
    if y_label:
        ax.set_ylabel(y_label)
    else:
        ax.set_ylabel(metric)

    # Title
    ax.set_title(title, y=legend_y - 0.07)

    # Save
    filename = filename or title.lower().replace(" ", "_")
    filename1 = f"{filename}.pdf"
    filename2 = f"{filename}.png"

    # Crop
    plt.subplots_adjust(**adjust)
    # plt.tight_layout()

    plt.savefig(os.path.join(OUTPUT_DIR, filename1), pad_inches=0.05, dpi=300)
    printc("Saved to path: ", os.path.join(OUTPUT_DIR, filename1))
    plt.savefig(os.path.join(OUTPUT_DIR, filename2), pad_inches=0.05, dpi=300)


ab3_ids = [
    # Group 1: Laplace Mechanism
    [32, 7, 33],
    # Group 2: VMF Mechanism
    [34, 35, 36],
    # Group 3: Gaussian Mechanism
    [37, 38, 39],
]
ab3_ids = [[f"{i}p" for i in group] for group in ab3_ids]  # Measurements only for Porto

ab3_group_names = [r"Laplace", r"VMF", r"Gaussian"]

ab3_labels = [
    r"$d_{out} = 4$",
    r"$d_{out} = 8$",
    r"$d_{out} = 256$",
]

adjust = {"left": 0.12, "right": 0.995, "top": 0.75, "bottom": 0.21}
create_bar_plot_ablation_III(
    ab3_ids, keys.SWD_DISTRIBUTION, group_names=ab3_group_names, labels=ab3_labels,
    y_label="SWD", title="Ablation Study III: DP-Conditional Information Mechanisms",
    filename="ablation_study_iii",
    group_offset=-0.002, adjust=adjust)